# M2 Backbones: MLP vs Mesh GNN

This notebook is the primary Colab workflow for the HemoMesh backbone milestone. It trains a dense per-node MLP baseline and a message-passing mesh GNN on the Suk coronary mesh dataset, then writes report-ready comparison artifacts under `results/`.

The workflow uses the modern project training stack (`torch` + `torch-geometric`) and does not depend on the legacy upstream GEM-GCN environment.

In [ ]:
from pathlib import Path

repo_dir = Path("/content/HemoMesh")
if not repo_dir.exists():
    !git clone https://github.com/Lawson-Darrow/HemoMesh.git /content/HemoMesh
%cd /content/HemoMesh
!git pull --ff-only

In [ ]:
!pip install -q -e ".[dev,analysis,torch]"

import torch
import torch_geometric

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyG: {torch_geometric.__version__}")

In [ ]:
from pathlib import Path

single_db = Path("vessel-datasets/stead/single/raw/database.hdf5")
bifurcating_db = Path("vessel-datasets/stead/bifurcating/raw/database.hdf5")
if not (single_db.exists() and bifurcating_db.exists()):
    !bash scripts/download_data.sh .
else:
    print("Dataset already present.")

In [ ]:
!mkdir -p results/artifacts
!hemomesh-inspect vessel-datasets/stead/single/raw/database.hdf5 \
  --subset single \
  --output results/artifacts/m1_single_dataset_inspection.json
!hemomesh-inspect vessel-datasets/stead/bifurcating/raw/database.hdf5 \
  --subset bifurcating \
  --output results/artifacts/m1_bifurcating_dataset_inspection.json

## Train The Dense Baseline

The MLP sees only per-node features. It is the dense baseline that the geometric model must beat.

In [ ]:
!hemomesh-train --config configs/m2_mlp.yaml

## Train The Mesh GNN

The MeshGNN uses the same node features as the MLP plus message passing over face-derived mesh edges.

In [ ]:
!hemomesh-train --config configs/m2_mesh_gnn.yaml

## Build Report Artifacts

This cell combines the two run summaries into a comparison table and a small figure for the report and presentation.

In [ ]:
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt

runs = [
    ("MLP", Path("results/artifacts/m2_mlp_summary.json")),
    ("MeshGNN", Path("results/artifacts/m2_mesh_gnn_summary.json")),
]
rows = []
for label, path in runs:
    payload = json.loads(path.read_text())
    metrics = payload["metrics"]
    rows.append(
        {
            "model": label,
            "test_wss_approximation_error_mean": metrics[
                "test_wss_approximation_error_mean"
            ],
            "test_wss_approximation_error_ci_low": metrics[
                "test_wss_approximation_error_ci_low"
            ],
            "test_wss_approximation_error_ci_high": metrics[
                "test_wss_approximation_error_ci_high"
            ],
            "test_wss_rmse_mean": metrics["test_wss_rmse_mean"],
            "test_wss_cosine_similarity_mean": metrics[
                "test_wss_cosine_similarity_mean"
            ],
            "test_pressure_rmse_mean": metrics["test_pressure_rmse_mean"],
        }
    )

Path("results/tables").mkdir(parents=True, exist_ok=True)
Path("results/figures").mkdir(parents=True, exist_ok=True)
comparison_csv = Path("results/tables/m2_backbone_comparison.csv")
with comparison_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

labels = [row["model"] for row in rows]
means = [row["test_wss_approximation_error_mean"] for row in rows]
errors = [
    [
        row["test_wss_approximation_error_mean"]
        - row["test_wss_approximation_error_ci_low"]
        for row in rows
    ],
    [
        row["test_wss_approximation_error_ci_high"]
        - row["test_wss_approximation_error_mean"]
        for row in rows
    ],
]
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(labels, means, yerr=errors, capsize=5, color=["#6b7280", "#2563eb"])
ax.set_ylabel("WSS approximation error")
ax.set_title("M2 backbone comparison")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
figure_path = Path("results/figures/m2_backbone_comparison.svg")
fig.savefig(figure_path)
plt.show()

print(f"Wrote {comparison_csv}")
print(f"Wrote {figure_path}")
rows